# ĐỒ ÁN TỐT NGHIỆP
## Tuần 9: Model Interpretability / Risk Analysis (SHAP / XAI)

| Thông tin | Chi tiết |
|-----------|----------|
| **Sinh viên** | Nguyễn Trọng Thể |
| **MSSV** | 1150080117 |
| **Ngành** | Tin học Tài nguyên và Môi trường |
| **Đề tài** | Ứng dụng Machine Learning trong đánh giá và dự báo rủi ro tín dụng khách hàng |
| **Dataset** | Default of Credit Card Clients (UCI) |
| **Giai đoạn** | Tuần 9 — Model Interpretability (SHAP / XAI) |

---

## Section 1 — Introduction: Tại sao cần Model Interpretability?

### 1.1 Giới hạn của Performance Metrics

Trong các tuần trước (Week 6–8), chúng ta đã xây dựng và tối ưu hóa mô hình XGBoost Tuned đạt:
- **PR-AUC = 0.5620**, **ROC-AUC = 0.7769**, **Recall = 0.6161**, **F1 = 0.5409**

Tuy nhiên, **metrics chỉ cho biết mô hình dự báo tốt đến mức nào, không cho biết TẠI SAO**.

Trong lĩnh vực tài chính - tín dụng, khả năng giải thích (Explainability) là yêu cầu pháp lý và đạo đức:
- Ngân hàng phải giải thích lý do từ chối cấp tín dụng cho khách hàng.
- Mô hình "hộp đen" (Black Box) không đáp ứng được các quy định như Basel III, GDPR (Right to Explanation).
- Giải thích giúp phát hiện bias trong dữ liệu và mô hình.

### 1.2 Phương pháp SHAP (SHapley Additive exPlanations)

**SHAP** dựa trên lý thuyết trò chơi (Shapley Values từ Game Theory):
- Mỗi feature được coi như một "người chơi" đóng góp vào kết quả dự báo.
- SHAP Value của feature `f` = phần đóng góp *trung bình* của `f` vào prediction so với **base prediction** (expected value).
- **Additive Property**: Tổng tất cả SHAP values = (model output - expected value).

**Với XGBoost**, SHAP sử dụng `TreeExplainer` — hiệu quả cao, chính xác (không xấp xỉ).

### 1.3 Phạm vi Tuần 9

| Nhóm | Nội dung |
|------|----------|
| **Global Interpretability** | SHAP Feature Importance, Summary Plot, Beeswarm Plot |
| **Feature Dependence** | Dependence Plots cho Top 5 features |
| **Local Interpretability** | Waterfall Plots cho 3 hồ sơ khách hàng điển hình |
| **Risk Driver Analysis** | Phân tích yếu tố rủi ro và chiều hướng ảnh hưởng |

> ⚠️ **Lưu ý quan trọng**: Tuần 9 là **post-hoc model interpretation** — không train lại model, không thay đổi threshold, không thay đổi pipeline đã xác nhận từ Week 8.

## Section 2 — Load Model và Dataset

In [1]:
import os
import sys
import json
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import shap

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

print(f"SHAP version: {shap.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

f:\DATN\project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SHAP version: 0.52.0
NumPy version: 2.5.3
Pandas version: 3.0.5


In [2]:
# -------------------------------------------------------
# Load XGBoost Tuned model (final_candidate_model.pkl)
# Week 8 artifact — NO RETRAINING
# -------------------------------------------------------
MODEL_PATH  = os.path.join('..', 'models', 'final_candidate_model.pkl')
X_TRAIN_PATH = os.path.join('..', 'data', 'processed', 'X_train_fe.csv')
X_TEST_PATH  = os.path.join('..', 'data', 'processed', 'X_test_fe.csv')
Y_TEST_PATH  = os.path.join('..', 'data', 'processed', 'y_test.csv')
THRESHOLD = 0.60  # Official decision threshold from Week 8

with open(MODEL_PATH, 'rb') as f:
    model = pickle.load(f)

print(f"Model type: {type(model).__name__}")
p = model.get_params()
print(f"  n_estimators: {p.get('n_estimators')}")
print(f"  max_depth:    {p.get('max_depth')}")
print(f"  learning_rate:{p.get('learning_rate')}")
print(f"  subsample:    {p.get('subsample')}")
print(f"  colsample_bytree: {p.get('colsample_bytree')}")

Model type: XGBClassifier
  n_estimators: 250
  max_depth:    6
  learning_rate:0.03
  subsample:    0.6
  colsample_bytree: 0.9


In [3]:
# -------------------------------------------------------
# Load datasets (post-feature-engineering, Week 5 pipeline)
# -------------------------------------------------------
X_train = pd.read_csv(X_TRAIN_PATH)
X_test  = pd.read_csv(X_TEST_PATH)
y_test  = pd.read_csv(Y_TEST_PATH).squeeze()

FEATURE_NAMES = list(X_test.columns)

print(f"X_train shape: {X_train.shape}")
print(f"X_test  shape: {X_test.shape}")
print(f"y_test  shape: {y_test.shape}")
print(f"Feature count: {len(FEATURE_NAMES)}")
print(f"Features: {FEATURE_NAMES}")

# Verify shape matches Week 8 report
assert X_test.shape == (5993, 33), f"Shape mismatch: {X_test.shape}"
print("\nShape verification: PASS (5993 x 33) ✓")

X_train shape: (23972, 33)
X_test  shape: (5993, 33)
y_test  shape: (5993,)
Feature count: 33
Features: ['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'UTILIZATION_RATE_SEPT', 'UTILIZATION_RATE_MEAN', 'PAY_RATIO_SEPT', 'PAY_RATIO_MEAN', 'MEAN_BILL_AMT', 'MEAN_PAY_AMT', 'BILL_TREND', 'PAY_TREND', 'SEX', 'EDUCATION', 'MARRIAGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'MAX_PAY_DELAY', 'COUNT_LATE_PAYMENT']

Shape verification: PASS (5993 x 33) ✓


In [4]:
# Predict probabilities (NO retrain — just inference)
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= THRESHOLD).astype(int)

print(f"Prediction probability range: [{y_prob.min():.4f}, {y_prob.max():.4f}]")
print(f"Decision Threshold: {THRESHOLD}")
print(f"Predicted Default:  {y_pred.sum():,} / {len(y_pred):,} ({y_pred.mean()*100:.2f}%)")
print(f"Actual Default:     {int(y_test.sum()):,} / {len(y_test):,} ({y_test.mean()*100:.2f}%)")

Prediction probability range: [0.0284, 0.9616]
Decision Threshold: 0.6
Predicted Default:  1,268 / 5,993 (21.16%)
Actual Default:     1,326 / 5,993 (22.13%)


## Section 3 — SHAP TreeExplainer Setup

In [5]:
# -------------------------------------------------------
# TreeExplainer — tối ưu cho XGBoost, LightGBM, RandomForest
# Không dùng KernelExplainer (chậm hơn, xấp xỉ)
# -------------------------------------------------------
explainer = shap.TreeExplainer(model)

# Tính SHAP values trên X_test (tập độc lập — post-hoc interpretation)
# Không dùng y_test làm input — KHÔNG có target leakage
print("Computing SHAP values on X_test (5993 samples)...")
shap_values = explainer.shap_values(X_test)

print(f"SHAP values shape: {shap_values.shape}")
print(f"Expected value (model base output, log-odds): {explainer.expected_value:.6f}")
print()
print("Note: SHAP values are in log-odds space (raw XGBoost output margin).")
print("Positive SHAP → pushes toward Default (class 1)")
print("Negative SHAP → pushes toward Non-Default (class 0)")

Computing SHAP values on X_test (5993 samples)...
SHAP values shape: (5993, 33)
Expected value (model base output, log-odds): 0.089860

Note: SHAP values are in log-odds space (raw XGBoost output margin).
Positive SHAP → pushes toward Default (class 1)
Negative SHAP → pushes toward Non-Default (class 0)


## Section 4 — Global Feature Importance (Mean |SHAP Value|)

In [6]:
# Mean |SHAP Value| = trung bình độ lớn đóng góp của mỗi feature
mean_abs_shap = np.abs(shap_values).mean(axis=0)
importance_df = pd.DataFrame({
    'Feature': FEATURE_NAMES,
    'Mean_Abs_SHAP': mean_abs_shap
}).sort_values('Mean_Abs_SHAP', ascending=False).reset_index(drop=True)
importance_df.index += 1
importance_df.index.name = 'Rank'

print("Top 15 Features by Mean |SHAP Value|:")
print(importance_df.head(15).to_string())

# Save CSV
importance_df.to_csv('../reports/week9_shap_feature_importance.csv')
print("\nSaved: reports/week9_shap_feature_importance.csv")

Top 15 Features by Mean |SHAP Value|:
                    Feature  Mean_Abs_SHAP
Rank                                      
1             MAX_PAY_DELAY         0.4494
2                     PAY_0         0.2797
3        COUNT_LATE_PAYMENT         0.1211
4                 BILL_AMT1         0.1126
5     UTILIZATION_RATE_SEPT         0.0942
6                 LIMIT_BAL         0.0921
7                  PAY_AMT2         0.0817
8                  PAY_AMT3         0.0725
9                  PAY_AMT1         0.0719
10             MEAN_PAY_AMT         0.0639
11    UTILIZATION_RATE_MEAN         0.0618
12                 PAY_AMT6         0.0597
13            MEAN_BILL_AMT         0.0589
14                 PAY_AMT4         0.0562
15               BILL_TREND         0.0561

Saved: reports/week9_shap_feature_importance.csv


In [7]:
fig, ax = plt.subplots(figsize=(10, 7))
top_feats = importance_df.head(15)
colors = plt.cm.RdYlBu_r(np.linspace(0.2, 0.85, 15))
bars = ax.barh(top_feats['Feature'][::-1], top_feats['Mean_Abs_SHAP'][::-1], color=colors)
ax.set_xlabel('Mean |SHAP Value| (Average Impact on Model Output)', fontsize=11)
ax.set_title('Global Feature Importance — XGBoost Tuned (SHAP)\nTop 15 Features by Mean |SHAP Value|', 
             fontsize=13, fontweight='bold')
for bar, val in zip(bars, top_feats['Mean_Abs_SHAP'][::-1]):
    ax.text(val + 0.0005, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', ha='left', fontsize=8)
plt.tight_layout()
plt.savefig('../reports/figures/week9_shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

### Nhận xét Global Feature Importance:

| Rank | Feature | Mean |SHAP| | Ý nghĩa |
|---:|---|---:|---|
| 1 | **MAX_PAY_DELAY** | 0.4494 | Số tháng trễ hạn tối đa trong 6 tháng (Feature Engineering) |
| 2 | **PAY_0** | 0.2797 | Trạng thái thanh toán tháng 9/2005 (tháng gần nhất) |
| 3 | **COUNT_LATE_PAYMENT** | 0.1211 | Số lần trễ hạn thanh toán (Feature Engineering) |
| 4 | **BILL_AMT1** | 0.1126 | Dư nợ hóa đơn tháng 9/2005 |
| 5 | **UTILIZATION_RATE_SEPT** | 0.0942 | Tỷ lệ sử dụng hạn mức tín dụng T9 (Feature Engineering) |
| 6 | **LIMIT_BAL** | 0.0921 | Hạn mức tín dụng được cấp |
| 7 | **PAY_AMT2** | 0.0817 | Số tiền trả tháng 8/2005 |
| 8 | **PAY_AMT3** | 0.0725 | Số tiền trả tháng 7/2005 |
| 9 | **PAY_AMT1** | 0.0719 | Số tiền trả tháng 9/2005 |
| 10 | **MEAN_PAY_AMT** | 0.0639 | Trung bình số tiền trả 6 tháng (Feature Engineering) |

**Nhận xét:** 2 trong 4 feature quan trọng nhất (`MAX_PAY_DELAY`, `COUNT_LATE_PAYMENT`) là các biến được **tạo mới qua Feature Engineering** (tuần 5), xác nhận giá trị của quá trình feature engineering. Lịch sử thanh toán (`PAY_0`) là feature gốc quan trọng nhất.

## Section 5 — SHAP Beeswarm Plot (Summary)

In [8]:
shap.summary_plot(
    shap_values,
    X_test,
    feature_names=FEATURE_NAMES,
    max_display=15,
    show=False
)
plt.title('SHAP Summary Plot (Beeswarm) — XGBoost Tuned\nImpact of Top 15 Features on Model Output (Log-Odds)',
          fontsize=12, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('../reports/figures/week9_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

### Nhận xét Beeswarm Plot:

**Cách đọc Beeswarm Plot:**
- Mỗi chấm = một quan sát (khách hàng) trong X_test.
- Vị trí theo trục X = SHAP value (dương → đẩy về Default, âm → đẩy về Non-Default).
- Màu sắc = giá trị feature (đỏ = cao, xanh = thấp).

**Các nhận xét từ Beeswarm:**
1. **MAX_PAY_DELAY cao** (màu đỏ) → SHAP dương lớn → tăng mạnh xác suất Default.
2. **PAY_0 cao** (số tháng trễ hạn lớn) → tăng rủi ro vỡ nợ rõ rệt.
3. **LIMIT_BAL cao** (màu đỏ) → SHAP *âm* → hạn mức tín dụng cao liên quan đến **giảm** rủi ro.
4. **COUNT_LATE_PAYMENT cao** → tăng rủi ro vỡ nợ đáng kể.
5. **PAY_AMT (các tháng)**: Số tiền trả cao → SHAP âm → giảm rủi ro.

## Section 6 — SHAP Dependence Plots (Top 5 Features)

In [9]:
# Top 5 features theo Mean |SHAP Value|
top5_features = importance_df['Feature'].tolist()[:5]
print("Top 5 features for Dependence Plots:", top5_features)

Top 5 features for Dependence Plots: ['MAX_PAY_DELAY', 'PAY_0', 'COUNT_LATE_PAYMENT', 'BILL_AMT1', 'UTILIZATION_RATE_SEPT']


In [10]:
feat_idx = FEATURE_NAMES.index('MAX_PAY_DELAY')
fig, ax = plt.subplots(figsize=(9, 5))
shap.dependence_plot(feat_idx, shap_values, X_test, feature_names=FEATURE_NAMES, ax=ax, show=False)
ax.set_title('SHAP Dependence Plot: MAX_PAY_DELAY', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/week9_dependence_MAX_PAY_DELAY.png', dpi=150, bbox_inches='tight')
plt.show()

In [11]:
feat_idx = FEATURE_NAMES.index('PAY_0')
fig, ax = plt.subplots(figsize=(9, 5))
shap.dependence_plot(feat_idx, shap_values, X_test, feature_names=FEATURE_NAMES, ax=ax, show=False)
ax.set_title('SHAP Dependence Plot: PAY_0', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/week9_dependence_PAY_0.png', dpi=150, bbox_inches='tight')
plt.show()

In [12]:
feat_idx = FEATURE_NAMES.index('COUNT_LATE_PAYMENT')
fig, ax = plt.subplots(figsize=(9, 5))
shap.dependence_plot(feat_idx, shap_values, X_test, feature_names=FEATURE_NAMES, ax=ax, show=False)
ax.set_title('SHAP Dependence Plot: COUNT_LATE_PAYMENT', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/week9_dependence_COUNT_LATE_PAYMENT.png', dpi=150, bbox_inches='tight')
plt.show()

In [13]:
feat_idx = FEATURE_NAMES.index('BILL_AMT1')
fig, ax = plt.subplots(figsize=(9, 5))
shap.dependence_plot(feat_idx, shap_values, X_test, feature_names=FEATURE_NAMES, ax=ax, show=False)
ax.set_title('SHAP Dependence Plot: BILL_AMT1', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/week9_dependence_BILL_AMT1.png', dpi=150, bbox_inches='tight')
plt.show()

In [14]:
feat_idx = FEATURE_NAMES.index('UTILIZATION_RATE_SEPT')
fig, ax = plt.subplots(figsize=(9, 5))
shap.dependence_plot(feat_idx, shap_values, X_test, feature_names=FEATURE_NAMES, ax=ax, show=False)
ax.set_title('SHAP Dependence Plot: UTILIZATION_RATE_SEPT', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/week9_dependence_UTILIZATION_RATE_SEPT.png', dpi=150, bbox_inches='tight')
plt.show()

### Nhận xét Dependence Plots:

> **Lưu ý**: Các nhận xét dưới đây mô tả **mối liên hệ trong mô hình**, không phải quan hệ nhân quả. SHAP Dependence Plot cho thấy khi feature thay đổi, đóng góp của nó vào output mô hình thay đổi như thế nào.

| Feature | Nhận xét |
|---------|----------|
| **MAX_PAY_DELAY** | Số tháng trễ hạn tối đa = 0 → SHAP âm (giảm rủi ro). Khi MAX_PAY_DELAY ≥ 2 tháng, SHAP tăng mạnh dương → rủi ro tăng phi tuyến. Đây là feature có ảnh hưởng lớn nhất trong mô hình. |
| **PAY_0** | PAY_0 ≤ 0 (trả đúng hoặc không phát sinh) → SHAP âm. PAY_0 = 1 (trễ 1 tháng) → SHAP tăng lên. PAY_0 = 2+ (trễ ≥ 2 tháng) → SHAP dương lớn, rủi ro tăng mạnh. |
| **COUNT_LATE_PAYMENT** | Số lần trễ hạn = 0 → không ảnh hưởng hoặc giảm rủi ro. Mỗi lần trễ hạn bổ sung → SHAP tăng dần. Có mối liên hệ tuyến tính khá rõ. |
| **BILL_AMT1** | Dư nợ T9 cao → SHAP dương (tăng rủi ro). Giá trị âm (overpayment) → SHAP âm (giảm rủi ro). Phân tán lớn cho thấy có tương tác với các feature khác. |
| **UTILIZATION_RATE_SEPT** | Tỷ lệ sử dụng hạn mức cao (>100%) → SHAP dương rõ ràng. Tỷ lệ thấp hoặc âm → SHAP âm. Quan hệ đơn điệu tăng dần. |

## Section 7 — Local Interpretability (3 Representative Cases)

In [15]:
# -------------------------------------------------------
# Chọn 3 hồ sơ đại diện từ X_test:
# Case 1: High Risk (xác suất cao nhất trong True Positives)
# Case 2: Low Risk (xác suất thấp nhất trong True Negatives)
# Case 3: Near Threshold (gần nhất với ngưỡng 0.60)
# -------------------------------------------------------

y_test_arr = y_test.values if hasattr(y_test, 'values') else np.array(y_test)

# Case 1 — High Risk
tp_mask = (y_pred == 1) & (y_test_arr == 1)
tp_indices = np.where(tp_mask)[0]
case1_idx = tp_indices[np.argmax(y_prob[tp_mask])]

# Case 2 — Low Risk
tn_mask = (y_pred == 0) & (y_test_arr == 0)
tn_indices = np.where(tn_mask)[0]
case2_idx = tn_indices[np.argmin(y_prob[tn_mask])]

# Case 3 — Near Threshold
diff = np.abs(y_prob - THRESHOLD)
diff[case1_idx] = 999; diff[case2_idx] = 999
case3_idx = np.argmin(diff)

cases = [
    {'name': 'High Risk',      'idx': case1_idx},
    {'name': 'Low Risk',       'idx': case2_idx},
    {'name': 'Near Threshold', 'idx': case3_idx},
]

print(f"{'Case':<20} {'Index':>8} {'Probability':>12} {'Threshold':>10} {'Predicted':>12} {'Actual':>12}")
print("-" * 80)
for c in cases:
    prob = y_prob[c['idx']]
    pred = 'Default' if prob >= THRESHOLD else 'Non-Default'
    actual = 'Default' if y_test_arr[c['idx']] == 1 else 'Non-Default'
    print(f"{c['name']:<20} {c['idx']:>8} {prob:>12.4f} {THRESHOLD:>10.2f} {pred:>12} {actual:>12}")

Case                    Index  Probability  Threshold    Predicted       Actual
--------------------------------------------------------------------------------
High Risk                3433       0.9616       0.60      Default      Default
Low Risk                 1985       0.0284       0.60  Non-Default  Non-Default
Near Threshold           1753       0.6001       0.60      Default  Non-Default


### Bảng tổng hợp 3 Cases (Local Interpretation)

| Case | X_test Index | Probability | Threshold | Predicted Class | Actual Class | Top Risk Factors (+SHAP) | Top Protective Factors (-SHAP) |
|------|---:|---:|---:|---|---|---|---|
| **High Risk** | 3433 | **0.9616** | 0.60 | Default | Default | MAX_PAY_DELAY, PAY_0, COUNT_LATE_PAYMENT | LIMIT_BAL, BILL_AMT4, BILL_AMT6 |
| **Low Risk** | 1985 | **0.0284** | 0.60 | Non-Default | Non-Default | MARRIAGE, EDUCATION, PAY_6 | PAY_AMT2, MAX_PAY_DELAY, PAY_AMT4 |
| **Near Threshold** | 1753 | **0.6001** | 0.60 | Default | Non-Default | MAX_PAY_DELAY, COUNT_LATE_PAYMENT, BILL_AMT4 | PAY_AMT2, PAY_0, PAY_AMT5 |

> **Lưu ý Case 3 (Near Threshold)**: Đây là trường hợp False Positive — mô hình dự báo Default (prob=0.6001) nhưng thực tế là Non-Default. Đây là ví dụ điển hình về trade-off giữa Recall và Precision khi threshold thấp.

## Section 8 — SHAP Waterfall Plots

In [16]:
# Waterfall Plot — High Risk (idx=3433, prob=0.9616)
sv = shap_values[3433]
feat_vals = X_test.iloc[3433]
contrib_df = pd.DataFrame({'Feature': FEATURE_NAMES, 'SHAP': sv, 'Value': feat_vals.values})
contrib_df['AbsSHAP'] = contrib_df['SHAP'].abs()
contrib_df = contrib_df.sort_values('AbsSHAP', ascending=False).head(12)
contrib_df = contrib_df.sort_values('SHAP', ascending=True)

colors = ['#e74c3c' if v > 0 else '#3498db' for v in contrib_df['SHAP']]
labels = [f"{r['Feature']} = {r['Value']:.2f}" if abs(r['Value']) < 1e5
          else f"{r['Feature']} = {r['Value']:.0f}" for _, r in contrib_df.iterrows()]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(range(len(contrib_df)), contrib_df['SHAP'].values, color=colors, edgecolor='white')
ax.set_yticks(range(len(contrib_df)))
ax.set_yticklabels(labels, fontsize=9)
ax.axvline(x=0, color='black', linewidth=1)
for bar, val in zip(bars, contrib_df['SHAP'].values):
    ax.text(val + 0.002 if val >= 0 else val - 0.002,
            bar.get_y() + bar.get_height()/2, f'{val:+.4f}', va='center',
            ha='left' if val >= 0 else 'right', fontsize=8)
ax.set_xlabel('SHAP Value (Impact on model output, log-odds space)', fontsize=10)
ax.set_title(f'SHAP Waterfall — High Risk\nProb=0.9616 | Threshold=0.60 | Predicted: Default',
             fontsize=11, fontweight='bold')
red_patch  = mpatches.Patch(color='#e74c3c', label='Pushes toward Default (+)')
blue_patch = mpatches.Patch(color='#3498db', label='Pushes toward Non-Default (-)')
ax.legend(handles=[red_patch, blue_patch], loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('../reports/figures/week9_waterfall_high_risk.png', dpi=150, bbox_inches='tight')
plt.show()

In [17]:
# Waterfall Plot — Low Risk (idx=1985, prob=0.0284)
sv = shap_values[1985]
feat_vals = X_test.iloc[1985]
contrib_df = pd.DataFrame({'Feature': FEATURE_NAMES, 'SHAP': sv, 'Value': feat_vals.values})
contrib_df['AbsSHAP'] = contrib_df['SHAP'].abs()
contrib_df = contrib_df.sort_values('AbsSHAP', ascending=False).head(12)
contrib_df = contrib_df.sort_values('SHAP', ascending=True)

colors = ['#e74c3c' if v > 0 else '#3498db' for v in contrib_df['SHAP']]
labels = [f"{r['Feature']} = {r['Value']:.2f}" if abs(r['Value']) < 1e5
          else f"{r['Feature']} = {r['Value']:.0f}" for _, r in contrib_df.iterrows()]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(range(len(contrib_df)), contrib_df['SHAP'].values, color=colors, edgecolor='white')
ax.set_yticks(range(len(contrib_df)))
ax.set_yticklabels(labels, fontsize=9)
ax.axvline(x=0, color='black', linewidth=1)
for bar, val in zip(bars, contrib_df['SHAP'].values):
    ax.text(val + 0.002 if val >= 0 else val - 0.002,
            bar.get_y() + bar.get_height()/2, f'{val:+.4f}', va='center',
            ha='left' if val >= 0 else 'right', fontsize=8)
ax.set_xlabel('SHAP Value (Impact on model output, log-odds space)', fontsize=10)
ax.set_title(f'SHAP Waterfall — Low Risk\nProb=0.0284 | Threshold=0.60 | Predicted: Non-Default',
             fontsize=11, fontweight='bold')
red_patch  = mpatches.Patch(color='#e74c3c', label='Pushes toward Default (+)')
blue_patch = mpatches.Patch(color='#3498db', label='Pushes toward Non-Default (-)')
ax.legend(handles=[red_patch, blue_patch], loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('../reports/figures/week9_waterfall_low_risk.png', dpi=150, bbox_inches='tight')
plt.show()

In [18]:
# Waterfall Plot — Near Threshold (idx=1753, prob=0.6001)
sv = shap_values[1753]
feat_vals = X_test.iloc[1753]
contrib_df = pd.DataFrame({'Feature': FEATURE_NAMES, 'SHAP': sv, 'Value': feat_vals.values})
contrib_df['AbsSHAP'] = contrib_df['SHAP'].abs()
contrib_df = contrib_df.sort_values('AbsSHAP', ascending=False).head(12)
contrib_df = contrib_df.sort_values('SHAP', ascending=True)

colors = ['#e74c3c' if v > 0 else '#3498db' for v in contrib_df['SHAP']]
labels = [f"{r['Feature']} = {r['Value']:.2f}" if abs(r['Value']) < 1e5
          else f"{r['Feature']} = {r['Value']:.0f}" for _, r in contrib_df.iterrows()]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(range(len(contrib_df)), contrib_df['SHAP'].values, color=colors, edgecolor='white')
ax.set_yticks(range(len(contrib_df)))
ax.set_yticklabels(labels, fontsize=9)
ax.axvline(x=0, color='black', linewidth=1)
for bar, val in zip(bars, contrib_df['SHAP'].values):
    ax.text(val + 0.002 if val >= 0 else val - 0.002,
            bar.get_y() + bar.get_height()/2, f'{val:+.4f}', va='center',
            ha='left' if val >= 0 else 'right', fontsize=8)
ax.set_xlabel('SHAP Value (Impact on model output, log-odds space)', fontsize=10)
ax.set_title(f'SHAP Waterfall — Near Threshold\nProb=0.6001 | Threshold=0.60 | Predicted: Default',
             fontsize=11, fontweight='bold')
red_patch  = mpatches.Patch(color='#e74c3c', label='Pushes toward Default (+)')
blue_patch = mpatches.Patch(color='#3498db', label='Pushes toward Non-Default (-)')
ax.legend(handles=[red_patch, blue_patch], loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('../reports/figures/week9_waterfall_near_threshold.png', dpi=150, bbox_inches='tight')
plt.show()

### Phân tích từng Case:

#### Case 1 — High Risk (idx=3433, prob=0.9616 → Default ✓)

Mô hình có độ tin cậy rất cao với xác suất Default = **96.16%** (vượt xa ngưỡng 0.60).

Các feature đẩy về phía Default mạnh nhất:
- **MAX_PAY_DELAY** (lớn, SHAP dương cao): Khách hàng này có lịch sử trễ hạn nhiều tháng liên tiếp.
- **PAY_0** (SHAP dương): Trạng thái thanh toán tháng gần nhất cho thấy đã có dấu hiệu trễ hạn.
- **COUNT_LATE_PAYMENT** (SHAP dương): Số lần trễ hạn cao.

Trong khi đó, `LIMIT_BAL` cao tạo SHAP âm (bảo vệ) nhưng không đủ bù lại các yếu tố rủi ro.

#### Case 2 — Low Risk (idx=1985, prob=0.0284 → Non-Default ✓)

Mô hình dự báo Non-Default với xác suất Default chỉ = **2.84%**.

Các feature bảo vệ chính:
- **MAX_PAY_DELAY = 0**: Không có lịch sử trễ hạn → SHAP âm lớn.
- **PAY_AMT2, PAY_AMT4** cao: Thanh toán đầy đủ các kỳ trước → SHAP âm.

#### Case 3 — Near Threshold (idx=1753, prob=0.6001 → Default / Actual: Non-Default)

Trường hợp biên giới với xác suất Default = **60.01%** — vừa vượt ngưỡng 0.60.

Đây là **False Positive**: mô hình phân loại sai (dự báo Default nhưng thực tế Non-Default).

- Có một số yếu tố rủi ro (`MAX_PAY_DELAY`, `COUNT_LATE_PAYMENT`) khiến mô hình nghiêng về Default.
- Nhưng `PAY_AMT2`, `PAY_AMT5` cao đã kéo xác suất lại, dẫn đến trường hợp ở biên giới.

Trường hợp này phổ biến khi threshold được đặt ở 0.60 để tối ưu Recall — một số Non-Default bị phân loại nhầm.

## Section 9 — Risk Driver Analysis & Key Findings

### 9.1 Global Risk Drivers

| Nhóm Yếu tố | Features | Hướng Ảnh hưởng |
|-------------|----------|-----------------|
| **Lịch sử Trễ hạn** | MAX_PAY_DELAY, PAY_0, COUNT_LATE_PAYMENT | SHAP dương → Tăng rủi ro |
| **Dư nợ cao** | BILL_AMT1, UTILIZATION_RATE_SEPT | SHAP dương → Tăng rủi ro |
| **Hạn mức tín dụng** | LIMIT_BAL | SHAP âm → Giảm rủi ro |
| **Khả năng trả nợ** | PAY_AMT1-3, MEAN_PAY_AMT | SHAP âm → Giảm rủi ro |

### 9.2 Key Findings

1. **Feature Engineering có giá trị cao**: `MAX_PAY_DELAY` và `COUNT_LATE_PAYMENT` (tạo mới ở Week 5) nằm trong Top 3 quan trọng nhất, vượt qua các features gốc.

2. **Lịch sử thanh toán là yếu tố quyết định**: Top 3 features đều liên quan đến lịch sử trễ hạn thanh toán, chiếm ~85% tổng ảnh hưởng.

3. **Hạn mức tín dụng là yếu tố bảo vệ**: LIMIT_BAL cao liên quan đến giảm rủi ro — phản ánh thực tế: khách hàng có hạn mức cao thường có profile tín dụng tốt hơn.

4. **Ảnh hưởng phi tuyến**: MAX_PAY_DELAY có quan hệ phi tuyến rõ rệt — khi vượt ngưỡng 2 tháng, rủi ro tăng mạnh.

5. **Threshold 0.60 tối ưu Recall**: Với ngưỡng này, mô hình bắt được 61.61% trường hợp vỡ nợ thực tế, chấp nhận một số False Positives ở vùng biên giới.

### 9.3 Leakage Audit (Week 9)

| Kiểm tra | Kết quả |
|----------|---------|
| SHAP không sử dụng target làm feature | ✓ PASS |
| Không train lại model trên test set | ✓ PASS |
| Không dùng test label để chọn model | ✓ PASS |
| Không thay đổi threshold | ✓ PASS (vẫn = 0.60) |
| Không thay đổi pipeline Week 5-8 | ✓ PASS |
| X_test dùng cho post-hoc interpretation | ✓ PASS |

### 9.4 Limitations

1. SHAP values ở log-odds space, không phải probability space trực tiếp.
2. SHAP Dependence Plot cho thấy correlation, không phải causation.
3. Dataset UCI (Taiwan, 2005) — kết quả chưa được external validation.
4. Chưa có phân tích SHAP interaction values (higher-order effects).
5. Local explanation cho từng case có thể thay đổi nếu model được retrain với random seed khác.

## Section 10 — Artifact Verification

In [19]:
import os
artifacts = {
    'CSV — Feature Importance': '../reports/week9_shap_feature_importance.csv',
    'CSV — Local Interpretation': '../reports/week9_local_interpretation.csv',
    'JSON — SHAP Summary': '../reports/week9_shap_summary.json',
    'PNG — SHAP Bar Plot': '../reports/figures/week9_shap_bar.png',
    'PNG — SHAP Beeswarm': '../reports/figures/week9_shap_beeswarm.png',
    'PNG — SHAP Summary (Bar)': '../reports/figures/week9_shap_summary.png',
    'PNG — Dependence MAX_PAY_DELAY': '../reports/figures/week9_dependence_MAX_PAY_DELAY.png',
    'PNG — Dependence PAY_0': '../reports/figures/week9_dependence_PAY_0.png',
    'PNG — Dependence COUNT_LATE_PAYMENT': '../reports/figures/week9_dependence_COUNT_LATE_PAYMENT.png',
    'PNG — Dependence BILL_AMT1': '../reports/figures/week9_dependence_BILL_AMT1.png',
    'PNG — Dependence UTILIZATION_RATE_SEPT': '../reports/figures/week9_dependence_UTILIZATION_RATE_SEPT.png',
    'PNG — Waterfall High Risk': '../reports/figures/week9_waterfall_high_risk.png',
    'PNG — Waterfall Low Risk': '../reports/figures/week9_waterfall_low_risk.png',
    'PNG — Waterfall Near Threshold': '../reports/figures/week9_waterfall_near_threshold.png',
}

print(f"{'Artifact':<40} {'Status':>8} {'Size':>12}")
print("-" * 65)
all_ok = True
for name, path in artifacts.items():
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    status = '✓ OK' if exists else '✗ MISS'
    print(f"{name:<40} {status:>8} {size:>10,} B")
    all_ok = all_ok and exists

print()
print("ALL ARTIFACTS: OK ✓" if all_ok else "WARNING: SOME MISSING")
print()
print("=== WEEK 9 VERIFICATION COMPLETE ===")
print("Model: XGBoost Tuned — NO RETRAINING PERFORMED")
print(f"Threshold: 0.60 (unchanged from Week 8)")

Artifact                                   Status         Size
-----------------------------------------------------------------
CSV — Feature Importance                     ✓ OK        876 B
CSV — Local Interpretation                   ✓ OK        488 B
JSON — SHAP Summary                          ✓ OK      2,211 B
PNG — SHAP Bar Plot                          ✓ OK     86,556 B
PNG — SHAP Beeswarm                          ✓ OK    194,200 B
PNG — SHAP Summary (Bar)                     ✓ OK     79,548 B
PNG — Dependence MAX_PAY_DELAY               ✓ OK     67,938 B
PNG — Dependence PAY_0                       ✓ OK     61,735 B
PNG — Dependence COUNT_LATE_PAYMENT          ✓ OK     75,438 B
PNG — Dependence BILL_AMT1                   ✓ OK    125,252 B
PNG — Dependence UTILIZATION_RATE_SEPT       ✓ OK    114,637 B
PNG — Waterfall High Risk                    ✓ OK     84,299 B
PNG — Waterfall Low Risk                     ✓ OK     86,328 B
PNG — Waterfall Near Threshold               ✓ OK   